In [ ]:
import numpy as np

from gridcp.new_api.detector import GridDetector, DetectorState
from gridcp.new_api.scores import CUSUM
from gridcp.new_api.typing import ArrayLike

In [ ]:
def run_online_grid_detector(
    data: ArrayLike,
    detector: GridDetector,
    reset_on_alarm: bool = False,
) -> tuple[list[DetectorState], list[dict]]:
    """Run a configured GridDetector over a dataset sequentially.

    Parameters
    ----------
    data : ArrayLike
        Sequence of observations. Each element is passed as `x` to
        `detector.update`. For univariate data this is a 1D array of scalars;
        for multivariate data it should be a 2D array of shape (n_samples, n_features).
    detector : GridDetector
        A fully configured detector instance (with `score` and `threshold`
        set). State is initialized internally via `detector.init_state()`.
    reset_on_alarm : bool, optional
        If True, the detector state is reset to a fresh initial state immediately
        after an alarm is raised. This allows the detector to restart tracking
        from the next observation after each detected changepoint.
        Default is False.

    Returns
    -------
    states : list[DetectorState]
        State after each observation, including the initial state at index 0.
        When `reset_on_alarm=False`, `states[i]` is the state after processing
        `data[i - 1]`, so `states[0]` is the fresh initial state.
        When `reset_on_alarm=True`, the index correspondence is broken at each
        alarm: the state is reset before the next observation is processed.
    outputs : list[dict]
        One entry per observation. Each dict contains:
          - ``"index"``: time index (n_samples) when the alarm was raised.
          - ``"alarm"``: bool, True when ``max_score > threshold``.
          - ``"max_score"``: highest penalized score among active candidates.
          - ``"max_score_index"``: grid position of the highest-scoring candidate.
    """
    data = np.asarray(data)
    state = detector.init_state()
    states = [state]
    outputs = []
    for x in data:
        state, output = detector.update(state, x)
        states.append(state)
        outputs.append(output)
        if output["alarm"] and reset_on_alarm:
            state = detector.init_state()

    return states, outputs

In [ ]:
def demo(n_samples=100, n_features=1, reset_on_alarm=False):
    """Simulate toy univariate data with a known mean shift and run the grid detector."""
    rng = np.random.default_rng(seed=42)
    n_pre, n_post = n_samples // 2, n_samples // 2 + n_samples % 2
    data = np.concatenate(
        [
            rng.normal(loc=0.0, scale=1.0, size=(n_pre, n_features)),
            rng.normal(loc=5.0, scale=1.0, size=(n_post, n_features)),
        ]
    )
    print(f"Simulated {len(data)} observations with a mean shift at index {n_pre}.\n")

    score = CUSUM(n_features)
    detector = GridDetector(score=score, threshold=10.0)
    states, detector_outputs = run_online_grid_detector(data, detector, reset_on_alarm)

    alarms = [s for s in detector_outputs if s["alarm"]]
    if alarms:
        print(f"Detected {len(alarms)} alarm(s):")
        for s in alarms:
            print(
                f"  t={s['index']:>4d}  max_score={s['max_score']:.3f}  "
                f"max_score_index={s['max_score_index']}"
            )
    else:
        print("No alarms raised.")

    final = states[-1]
    print(f"\nFinal state: t={final.n_samples}")

In [4]:
demo(100)

Simulated 100 observations with a mean shift at index 50.

Detected 48 alarm(s):
  t=  53  max_score=10.451  max_score_index=50
  t=  54  max_score=10.494  max_score_index=49
  t=  55  max_score=13.123  max_score_index=49
  t=  56  max_score=15.461  max_score_index=49
  t=  57  max_score=18.117  max_score_index=49
  t=  58  max_score=22.733  max_score_index=49
  t=  59  max_score=24.486  max_score_index=49
  t=  60  max_score=28.283  max_score_index=49
  t=  61  max_score=28.919  max_score_index=49
  t=  62  max_score=31.031  max_score_index=49
  t=  63  max_score=33.610  max_score_index=49
  t=  64  max_score=36.563  max_score_index=49
  t=  65  max_score=39.560  max_score_index=49
  t=  66  max_score=42.554  max_score_index=49
  t=  67  max_score=44.222  max_score_index=49
  t=  68  max_score=45.709  max_score_index=49
  t=  69  max_score=48.518  max_score_index=49
  t=  70  max_score=50.155  max_score_index=49
  t=  71  max_score=50.640  max_score_index=49
  t=  72  max_score=51.257

In [ ]:
def benchmark(n_samples=1000, n_features=1, n_repeats=5, top_n=20):
    """Profile and time run_online_grid_detector to identify bottlenecks.

    Runs a Numba warm-up pass first (compilation overhead excluded from timing),
    then times `n_repeats` runs and profiles one representative run with cProfile.

    Parameters
    ----------
    n_samples : int
        Number of observations per run.
    n_features : int
        Observation dimensionality.
    n_repeats : int
        Number of timed repetitions (warm-up excluded).
    top_n : int
        Number of hottest functions to print from cProfile output.
    """
    import cProfile
    import pstats
    import io
    import timeit

    rng = np.random.default_rng(seed=0)
    n_pre = n_samples // 2
    n_post = n_samples - n_pre
    data = np.concatenate(
        [
            rng.normal(0.0, 1.0, size=(n_pre, n_features)),
            rng.normal(5.0, 1.0, size=(n_post, n_features)),
        ]
    )

    detector = GridDetector(score=CUSUM(n_features), threshold=10.0)

    # Warm up Numba JIT compilation - excluded from timing.
    print("Warming up Numba (first call compiles)...")
    run_online_grid_detector(data[: min(50, n_samples)], detector)
    print("Warm-up done.\n")

    # Timing
    def _run():
        run_online_grid_detector(data, detector)

    times = timeit.repeat(_run, number=1, repeat=n_repeats)
    print(
        f"Timing over {n_repeats} runs (n_samples={n_samples}, n_features={n_features}):"
    )
    print(
        f"  min={min(times) * 1000:.2f}ms  mean={sum(times) / len(times) * 1000:.2f}ms  max={max(times) * 1000:.2f}ms\n"
    )

    # cProfile of a single run
    pr = cProfile.Profile()
    pr.enable()
    run_online_grid_detector(data, detector)
    pr.disable()

    buf = io.StringIO()
    ps = pstats.Stats(pr, stream=buf).sort_stats("cumulative")
    ps.print_stats(top_n)
    print(f"cProfile output (top {top_n} by cumulative time):")
    print(buf.getvalue())

In [10]:
benchmark(n_samples=100000, n_features=1, n_repeats=5, top_n=20)

Warming up Numba (first call compiles)...
Warm-up done.

Timing over 5 runs (n_samples=100000, n_features=1):
  min=2089.44ms  mean=2212.64ms  max=2276.17ms

cProfile output (top 20 by cumulative time):
         5972755 function calls (5972750 primitive calls) in 4.438 seconds

   Ordered by: cumulative time
   List reduced from 154 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      2/1    0.130    0.065    4.416    4.416 C:\Users\tveten\AppData\Local\Temp\ipykernel_23108\2760363049.py:1(run_online_grid_detector)
   100000    0.311    0.000    4.281    0.000 C:\Users\tveten\local_projects\G-CHAD\gridcp\new_api\detector.py:106(update)
    99999    1.105    0.000    3.186    0.000 C:\Users\tveten\local_projects\G-CHAD\gridcp\new_api\scores\_mean_cusum.py:102(compute_penalized_scores)
    99999    1.550    0.000    1.827    0.000 c:\Users\tveten\local_projects\G-CHAD\.venv\Lib\site-packages\numpy\_core\shape_base.py:378(stack)
   1